In [ ]:
from transformers import PreTrainedTokenizerFast
import json
import os
from glob import glob
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict, Counter
import ipywidgets as widgets
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from tqdm import tqdm

base_path = "../data/eval_results/Pythia-12-8-128-dfs"
# base_path = "../data/eval_results/PAST-6-4-128-dfs"
val_data = "../data/sos_filtered/val1_b4_t30_n500000_dfs_filtered.json"

In [ ]:
def load_step_jsons(base_path):
    """
    Load JSON-formatted files from step folders, checking first for files without 
    extensions, then falling back to .json files if none found.
    
    Args:
        base_path (str): Path to the directory containing step folders
        
    Returns:
        list: List of tuples containing (step_number, json_data)
    """
    results = []
    
    # Get all step directories using glob
    step_dirs = glob(os.path.join(base_path, "step_*"))
    for step_dir in step_dirs:
        try:
            # Extract step number from directory name
            step_num = int(os.path.basename(step_dir).split('_')[1])
            
            # List all files in the directory
            files = os.listdir(step_dir)
            
            # First look for files without extensions
            json_files = [os.path.join(step_dir, f) for f in files if os.path.isfile(os.path.join(step_dir, f)) and '.' not in f]
            
            # If no extensionless files found, look for .json files
            if not json_files:
                json_files = glob(os.path.join(step_dir, "*.json"))
            
            if json_files:
                json_path = json_files[0]  # Now json_files already contains full paths
                
                # Read and parse JSON
                with open(json_path, 'r') as f:
                    data = json.load(f)
                    
                results.append((step_num, data))
        except ValueError as e:
            print(f"Error parsing step number from {step_dir}: {e}")
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON in step {step_num}: {e}")
        except Exception as e:
            print(f"Error processing {step_dir}: {e}")
    
    # Sort results by step number
    results.sort(key=lambda x: x[0])
    
    return results

jsons = load_step_jsons(base_path)

print(f"Loaded {len(jsons)} JSON files")

eval_data = []
with open(val_data, "r") as f:
    for line in f:
        if line.strip():
            try:
                eval_data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error parsing line: {e}")
                continue

def get_tokenizer(path):
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=path
    )
    tokenizer.eos_token = "[EOS]"
    tokenizer.unk_token = "[UNK]"
    tokenizer.pad_token = "[PAD]"
    tokenizer.mask_token = "[MASK]"
    tokenizer.bos_token = "[BOS]"
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


In [ ]:
eval_data[0]

In [ ]:
tokenizer = get_tokenizer("../tokenizer/tokenizer.json")

def compare_lists(list1, list2):
    # Get the length of the longer list
    max_length = max(len(list1), len(list2))
    
    # Initialize result list with zeros
    result = [0] * max_length
    
    # Compare elements at corresponding indices
    for i in range(min(len(list1), len(list2))):
        if list1[i] == list2[i]:
            result[i] = 1
            
    return result

def get_eval_ids_lst(eval_data, num_samples):
    ret = []
    for i, sample in enumerate(eval_data):
        if i < num_samples:
            sp = sample["search_path"].split(", ", 1)[1]
            encoded_2 = tokenizer.encode(sp, return_tensors='pt')
            ret.append(encoded_2.tolist()[0])
    return ret

eval_input_ids = get_eval_ids_lst(eval_data, 512)
full_dic = defaultdict(list)
for js in tqdm(jsons):
    ix = 0
    step_num = js[0]
    for traj in js[1]["trajectories"]:
        traj = traj.replace("[BOS] ", "").split(", ", 1)[1]
        encoded = tokenizer.encode(traj, return_tensors='pt')
        out_ids_lst = encoded.tolist()[0]
        result = compare_lists(out_ids_lst, eval_input_ids[ix])
        count_matches = result.count(1)
        match_indices = [i for i, value in enumerate(result) if value == 1]
        tokenized_out = tokenizer.tokenize(traj)
        match_toks = []
        for index in match_indices:
            match_toks.append(tokenized_out[index])

        full_dic[step_num].append((ix, count_matches, match_indices, match_toks))
        ix+=1



In [ ]:
txt = "ahoj , ahoj , ahoj"
sp = txt.split(", ", 1)[1]
sp

In [ ]:
aggr = []
for k, v in full_dic.items():
    all_matches = 0
    for i, res in enumerate(v):
        ix, num_matches, indices, tokens = res
        all_matches += num_matches
    aggr.append(all_matches/i)
print(aggr)

In [6]:
toks = []
for k, v in full_dic.items():
    toks_temp = []
    for i, res in enumerate(v):
        _, _, _, tokens = res
        toks_temp.append(tokens)
    toks.append(toks_temp)

In [ ]:
data=aggr
# Create figure and axis
plt.figure(figsize=(15, 6))

# Create bar plot
plt.bar(range(len(data)), data, color='skyblue', edgecolor='black')

# Customize the plot
plt.title('Bar Plot')
plt.xlabel('Index')
plt.ylabel('Value')

# Add grid for better readability
plt.grid(True, alpha=0.3, axis='y')

# Display the plot
plt.tight_layout()
plt.show()

In [8]:
ret = defaultdict(list)
for ix, json in enumerate(toks):
    dic = Counter()  # Use Counter directly instead of regular dict
    for lst in json:
        dic.update(Counter(lst))  # Update will sum the counts
    
    ret[ix].append(dict(dic))

In [ ]:
len(ret)

In [ ]:
ret[4]

In [ ]:
def plot_histogram(index):
    # Get the dictionary for the specified index
    data = ret[index][0]  # ret[index] is a list containing one dict
    
    # Create the histogram
    plt.figure(figsize=(15, 6))
    plt.bar(range(len(data)), list(data.values()))
    
    # Set the x-axis labels (tokens)
    plt.xticks(range(len(data)), list(data.keys()), rotation=45, ha='right')
    
    plt.title(f'Token Counts for Dictionary {index}')
    plt.xlabel('Tokens')
    plt.ylabel('Count')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


@interact(index=(0, 57))
def plot_interactive_histogram(index):
    plot_histogram(index)

In [ ]:
def create_highlighted_text(step_num, traj_idx):
    text = eval_data[traj_idx]["search_path"]
    text_lst = text.split(" ")
    
    # Access data exactly like your script
    _, _, match_indices, _ = full_dic[step_num][traj_idx]
    match_indices.sort()
    
    # Create HTML with highlights
    html_parts = []
    for idx, word in enumerate(text_lst):
        if idx in match_indices:
            html_parts.append(f'<span style="background-color: red; color: white;">{word}</span>')
        else:
            html_parts.append(word)
    
    highlighted_text = ' '.join(html_parts)
    return HTML(f'<div style="font-family: monospace; white-space: pre-wrap; font-size: 14px;">{highlighted_text}</div>')

def interactive_text_viewer():
    initial_step = list(full_dic.keys())[0]  # Set initial step to 8334
    if full_dic[0]:
        del full_dic[0]
    step_dropdown = widgets.Dropdown(
        options=sorted(full_dic.keys()),
        value=initial_step,  # Set initial value
        description='Step:',
        style={'description_width': 'initial'}
    )
    
    def update_traj_options(*args):
        traj_dropdown.options = range(len(full_dic[step_dropdown.value]))
    
    traj_dropdown = widgets.Dropdown(
        options=range(len(full_dic[initial_step])),  # Use initial_step here too
        description='Trajectory:',
        style={'description_width': 'initial'}
    )
    
    step_dropdown.observe(update_traj_options, 'value')
    
    output = widgets.Output()
    
    def update_display(*args):
        with output:
            output.clear_output()
            display(create_highlighted_text(
                step_dropdown.value,
                traj_dropdown.value
            ))
    
    step_dropdown.observe(update_display, 'value')
    traj_dropdown.observe(update_display, 'value')
    
    display(widgets.HBox([step_dropdown, traj_dropdown]))
    display(output)
    update_display()

# Run the viewer
interactive_text_viewer()

In [28]:
import pickle

# load pickle
with open('full_dic.pkl', 'rb') as f:
    full_dic = pickle.load(f)

with open('eval_data.pkl', 'rb') as s:
    eval_data = pickle.load(s)

In [ ]:
full_dic[6250]

In [38]:
sample_idx = 0
step_pre_idx = 0
step_after_idx = 1

step_pre_jump = list(full_dic.keys())[step_pre_idx]
prev = full_dic[step_pre_jump]
ix, _, pos_ids, _ = prev[sample_idx]

step_after_jump = list(full_dic.keys())[step_after_idx]
post = full_dic[step_after_jump]
_, _, new_pos_ids, _ = post[sample_idx]


In [ ]:
ix

In [ ]:
pos_ids_set = set(pos_ids)
new_pos_ids_set = set(new_pos_ids)

diff = new_pos_ids_set.difference(pos_ids_set)

diff = list(diff)

In [ ]:
text = eval_data[sample_idx]["search_path"][23:]
text

In [ ]:
import ipywidgets as widgets
from IPython.display import HTML, display

def create_highlighted_text(step_num, traj_idx):
    text = eval_data[traj_idx]["search_path"]
    text_lst = text.split(" ")
    
    # Access data exactly like your script
    _, _, match_indices, _ = full_dic[step_num][traj_idx]
    match_indices.sort()
    
    # Create HTML with highlights
    html_parts = []
    for idx, word in enumerate(text_lst):
        if idx in match_indices:
            html_parts.append(f'<span style="background-color: red; color: white;">{word}</span>')
        else:
            html_parts.append(word)
    
    highlighted_text = ' '.join(html_parts)
    return HTML(f'<div style="font-family: monospace; white-space: pre-wrap; font-size: 14px; width: 100%;">{highlighted_text}</div>')

def create_highlighted_text_diff(step_pre_num, pre_traj_idx, step_after_num, next_traj_idx):
    text = eval_data[next_traj_idx]["search_path"]
    text_lst = text.split(" ")

    # Get data for previous step
    _, _, pos_ids, _ = full_dic[step_pre_num][pre_traj_idx]
    # Get data for next step
    _, _, new_pos_ids, _ = full_dic[step_after_num][next_traj_idx]

    # Convert to sets and find difference
    pos_ids_set = set(pos_ids)
    new_pos_ids_set = set(new_pos_ids)
    diff = new_pos_ids_set.difference(pos_ids_set)

    # Sort the differences for consistent highlighting
    diff_idxs = sorted(list(diff))
    
    # Create HTML with highlights
    html_parts = []
    for idx, word in enumerate(text_lst):
        if idx in diff_idxs:
            html_parts.append(f'<span style="background-color: red; color: white;">{word}</span>')
        else:
            html_parts.append(word)
    
    highlighted_text = ' '.join(html_parts)
    return HTML(f'<div style="font-family: monospace; white-space: pre-wrap; font-size: 14px; width: 100%;">{highlighted_text}</div>')

def side_by_side_viewer():
    # Set fixed width for the entire container
    main_container = widgets.HTML(
        value='<div style="width: 100%; max-width: 100vw;">'
    )
    
    # Initialize step dropdowns
    initial_step = list(full_dic.keys())[0]
    if full_dic.get(0):
        del full_dic[0]
        
    left_step_dropdown = widgets.Dropdown(
        options=sorted(full_dic.keys()),
        value=initial_step,
        description='Left Step:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='95%')
    )
    
    right_step_dropdown = widgets.Dropdown(
        options=sorted(full_dic.keys()),
        value=initial_step,
        description='Right Step:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='95%')
    )
    
    # Initialize trajectory dropdowns
    left_traj_dropdown = widgets.Dropdown(
        options=range(len(full_dic[initial_step])),
        description='Left Trajectory:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='95%')
    )
    
    right_traj_dropdown = widgets.Dropdown(
        options=range(len(full_dic[initial_step])),
        description='Right Trajectory:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='95%')
    )
    
    # Create output widgets for displaying text
    left_output = widgets.Output(layout=widgets.Layout(width='100%'))
    right_output = widgets.Output(layout=widgets.Layout(width='100%'))
    
    def update_left_traj_options(*args):
        left_traj_dropdown.options = range(len(full_dic[left_step_dropdown.value]))
        update_right_display()  # Update right display when left changes
        
    def update_right_traj_options(*args):
        right_traj_dropdown.options = range(len(full_dic[right_step_dropdown.value]))
    
    def update_left_display(*args):
        with left_output:
            left_output.clear_output()
            display(create_highlighted_text(
                left_step_dropdown.value,
                left_traj_dropdown.value
            ))
        update_right_display()  # Update right display when left changes
    
    def update_right_display(*args):
        with right_output:
            right_output.clear_output()
            display(create_highlighted_text_diff(
                left_step_dropdown.value,
                left_traj_dropdown.value,
                right_step_dropdown.value,
                right_traj_dropdown.value
            ))
    
    # Connect observers
    left_step_dropdown.observe(update_left_traj_options, 'value')
    right_step_dropdown.observe(update_right_traj_options, 'value')
    
    left_step_dropdown.observe(update_left_display, 'value')
    left_traj_dropdown.observe(update_left_display, 'value')
    
    right_step_dropdown.observe(update_right_display, 'value')
    right_traj_dropdown.observe(update_right_display, 'value')
    
    # Create control panels for each side with fixed widths
    left_controls = widgets.VBox([left_step_dropdown, left_traj_dropdown], 
                               layout=widgets.Layout(width='50%'))
    right_controls = widgets.VBox([right_step_dropdown, right_traj_dropdown],
                                layout=widgets.Layout(width='50%'))
    controls = widgets.HBox([left_controls, right_controls],
                          layout=widgets.Layout(width='100%'))
    
    # Create panels for text display with fixed widths
    left_panel = widgets.VBox([
        widgets.HTML('<div style="border: 1px solid #ccc; padding: 10px; margin: 5px; width: 95%;">'),
        left_output,
        widgets.HTML('</div>')
    ], layout=widgets.Layout(width='50%'))
    
    right_panel = widgets.VBox([
        widgets.HTML('<div style="border: 1px solid #ccc; padding: 10px; margin: 5px; width: 95%;">'),
        right_output,
        widgets.HTML('</div>')
    ], layout=widgets.Layout(width='50%'))
    
    panels = widgets.HBox([left_panel, right_panel],
                         layout=widgets.Layout(width='100%'))
    
    # Display everything
    display(main_container)
    display(controls)
    display(panels)
    
    # Initial display
    update_left_display()
    update_right_display()

# Run the viewer
side_by_side_viewer()